# SASV: LFCC weighted fusion `α·s_asv + (1−α)·s_cm`

Reuses existing LFCC fusion score CSVs (no GPU re-score):

- `runs/ecapa_plus_lfcc_dev/scores_dev.csv`
- `runs/ecapa_plus_lfcc_eval/scores_eval.csv`

**Protocol**

1. Sweep `α ∈ [0, 1]` on **dev**; pick α that minimizes **SASV-EER**
2. Lock that α and evaluate **once** on **eval** (do not re-tune)

Baseline score-sum in notebooks 03/04 is `α = 0.5` only if scores are on similar scales;
those notebooks used `s_asv + s_cm` (unnormalized sum). Here we use a true convex combination.

Also reports the unnormalized sum `s_asv + s_cm` for comparison with your locked LFCC numbers.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "weighted_fusion_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

from experiment_lib import DEFAULT_SASV, RUNS_DIR, ensure_sasv_on_path
from weighted_fusion_lib import (
    eers_for_alpha,
    load_score_csv,
    save_weighted_run,
    sweep_alpha,
    weighted_scores,
)

ensure_sasv_on_path(DEFAULT_SASV)
from metrics import get_all_EERs

DEV_CSV = RUNS_DIR / "ecapa_plus_lfcc_dev" / "scores_dev.csv"
EVAL_CSV = RUNS_DIR / "ecapa_plus_lfcc_eval" / "scores_eval.csv"
print("dev csv:", DEV_CSV.exists(), DEV_CSV)
print("eval csv:", EVAL_CSV.exists(), EVAL_CSV)

## Knobs

- `N_GRID` — number of α points on `[0, 1]` (21 → step 0.05)
- Set `RUN_EVAL = True` only after you are happy with the locked α from **dev**

In [ ]:
N_GRID = 21
ALPHAS = np.linspace(0.0, 1.0, N_GRID)
RUN_EVAL = True  # apply locked α once on eval
CM_BACKEND = "lfcc"

## 1. Load **dev** LFCC scores

In [ ]:
s_asv_dev, s_cm_dev, keys_dev = load_score_csv(DEV_CSV)
print(f"dev trials: {len(keys_dev)}")
print(
    "s_asv range:",
    float(s_asv_dev.min()),
    "→",
    float(s_asv_dev.max()),
    "| s_cm range:",
    float(s_cm_dev.min()),
    "→",
    float(s_cm_dev.max()),
)

## 2. Reference: unnormalized sum `s_asv + s_cm` (notebook 03)

In [ ]:
sum_preds = (s_asv_dev + s_cm_dev).tolist()
sasv_sum, sv_sum, spf_sum = get_all_EERs(sum_preds, keys_dev)
ref_sum = {
    "fusion": "s_asv + s_cm",
    "sasv_eer_%": sasv_sum * 100,
    "sv_eer_%": sv_sum * 100,
    "spf_eer_%": spf_sum * 100,
}
ref_sum

## 3. Sweep α on **dev** (minimize SASV-EER)

In [ ]:
best_dev, sweep_rows = sweep_alpha(
    s_asv_dev,
    s_cm_dev,
    keys_dev,
    alphas=ALPHAS,
    sasv_root=DEFAULT_SASV,
)
LOCKED_ALPHA = float(best_dev["alpha"])
print("Locked α (min SASV-EER on dev):", LOCKED_ALPHA)
print(
    f"dev weighted  SASV={best_dev['sasv_eer_percent']:.4f}%  "
    f"SV={best_dev['sv_eer_percent']:.4f}%  SPF={best_dev['spf_eer_percent']:.4f}%"
)

# Show a few nearby alphas
print("\nα sweep (dev):")
for row in sweep_rows:
    mark = " <-- best" if row["alpha"] == LOCKED_ALPHA else ""
    print(
        f"  α={row['alpha']:.2f}  SASV={row['sasv_eer_percent']:7.4f}%  "
        f"SV={row['sv_eer_percent']:7.4f}%  SPF={row['spf_eer_percent']:7.4f}%{mark}"
    )

In [ ]:
dev_out = save_weighted_run(
    split="dev",
    alpha=LOCKED_ALPHA,
    s_asv=s_asv_dev,
    s_cm=s_cm_dev,
    keys=keys_dev,
    metrics=best_dev,
    source_csv=DEV_CSV,
)
print("Wrote", dev_out)
# Also save full sweep table
(dev_out / "alpha_sweep_dev.json").write_text(
    json.dumps({"locked_alpha": LOCKED_ALPHA, "sweep": sweep_rows}, indent=2),
    encoding="utf-8",
)

## 4. Locked **eval** (same α — do not re-tune)

Requires `runs/ecapa_plus_lfcc_eval/scores_eval.csv` from notebook `04`.

In [ ]:
if not RUN_EVAL:
    print("RUN_EVAL=False — skip eval")
else:
    s_asv_ev, s_cm_ev, keys_ev = load_score_csv(EVAL_CSV)
    print(f"eval trials: {len(keys_ev)} | locked α={LOCKED_ALPHA}")

    # Reference unnormalized sum on eval
    sasv_s, sv_s, spf_s = get_all_EERs((s_asv_ev + s_cm_ev).tolist(), keys_ev)
    print(
        f"eval sum (s_asv+s_cm)  SASV={sasv_s*100:.4f}%  "
        f"SV={sv_s*100:.4f}%  SPF={spf_s*100:.4f}%"
    )

    eval_metrics = eers_for_alpha(
        s_asv_ev, s_cm_ev, keys_ev, LOCKED_ALPHA, sasv_root=DEFAULT_SASV
    )
    print(
        f"eval weighted α={LOCKED_ALPHA:.2f}  "
        f"SASV={eval_metrics['sasv_eer_percent']:.4f}%  "
        f"SV={eval_metrics['sv_eer_percent']:.4f}%  "
        f"SPF={eval_metrics['spf_eer_percent']:.4f}%"
    )

    eval_out = save_weighted_run(
        split="eval",
        alpha=LOCKED_ALPHA,
        s_asv=s_asv_ev,
        s_cm=s_cm_ev,
        keys=keys_ev,
        metrics=eval_metrics,
        source_csv=EVAL_CSV,
    )
    # Persist locked α explicitly for the paper table
    (eval_out / "locked_alpha.json").write_text(
        json.dumps(
            {
                "locked_alpha": LOCKED_ALPHA,
                "tuned_on": "dev",
                "objective": "min SASV-EER",
                "eval_metrics": eval_metrics,
                "dev_metrics": best_dev,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Wrote", eval_out)

## Done

Report **eval** weighted EERs with the **dev-locked** α. Do not pick α from eval.

Outputs:

- `runs/ecapa_plus_lfcc_weighted_dev/`
- `runs/ecapa_plus_lfcc_weighted_eval/`